In [19]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime
import re

In [3]:
headers = {"User-Agent": "MyDataResearchProject-dev-1.0.0", "Accept": "application/ld+json"}

In [15]:
df = pd.read_csv("all_meetings_raw.csv")
df.head()

,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
0,0,eli/dl/event/MTG-PL-2024-01-15,Activity,2024-01-15,2024-01-15T23:00:00+01:00,MTG-PL-2024-01-15,"{'hu': '2024. január 15., hétfő', 'sv': 'Månda...",2024-01-15T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-15-PVCRE-ITM-14'...,['eli/dl/doc/OJQ-9-2024-01-15'],def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197579', 'person/125...","['person/99283', 'person/88882', 'person/12480...",581.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-15-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
1,1,eli/dl/event/MTG-PL-2024-01-16,Activity,2024-01-16,2024-01-16T23:00:00+01:00,MTG-PL-2024-01-16,"{'da': 'Tirsdag den 16. januar 2024', 'mt': ""I...",2024-01-16T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-16-PVCRE-ITM-11'...,['eli/dl/doc/OJQ-9-2024-01-16'],def/ep-activities/PLENARY_SITTING,"['person/36392', 'person/125030', 'person/1974...","['person/125214', 'person/197754', 'person/967...",636.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-16-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
2,2,eli/dl/event/MTG-PL-2024-01-17,Activity,2024-01-17,2024-01-17T23:00:00+01:00,MTG-PL-2024-01-17,"{'it': 'Mercoledì 17 gennaio 2024', 'en': 'Wed...",2024-01-17T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-17-VOT-ITM-93994...,['eli/dl/doc/OJQ-9-2024-01-17'],def/ep-activities/PLENARY_SITTING,"['person/198183', 'person/36392', 'person/9677...","['person/197607', 'person/197770', 'person/125...",637.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-17-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
3,3,eli/dl/event/MTG-PL-2024-01-18,Activity,2024-01-18,2024-01-18T23:00:00+01:00,MTG-PL-2024-01-18,"{'lt': 'Ketvirtadienis, 2024 m. sausio 18 d.',...",2024-01-18T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-18-VOT-ITM-93993...,['eli/dl/doc/OJQ-9-2024-01-18'],def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197478', 'person/132...","['person/4746', 'person/197772', 'person/24457...",586.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-18-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
4,4,eli/dl/event/MTG-PL-2024-01-25,Activity,2024-01-25,2024-01-25T23:00:00+01:00,MTG-PL-2024-01-25,"{'de': 'Donnerstag, 25. Januar 2024', 'it': 'G...",2024-01-25T01:00:00+01:00,"['eli/dl/event/MTG-PL-2024-01-25-PVCRE-ITM-5',...",['eli/dl/doc/OJQ-9-2024-01-25'],def/ep-activities/PLENARY_SITTING,"['person/197579', 'person/124770', 'person/197...","['person/197840', 'person/197782', 'person/197...",340.0,org/ep-9,"['eli/dl/doc/CRE-9-2024-01-25', 'eli/dl/doc/PV...",http://publications.europa.eu/resource/authori...,NaN


In [16]:
#We will add more raw meetings to the all_meetings_raw csv for data to make comparisons with
json_files = [
    "more_years_raw_all_meetings_jsons/meetings_2014.json",
    "more_years_raw_all_meetings_jsons/meetings_2015.json",
    "more_years_raw_all_meetings_jsons/meetings_2016.json",
    "more_years_raw_all_meetings_jsons/meetings_2017.json",
    "more_years_raw_all_meetings_jsons/meetings_2018.json",
    "more_years_raw_all_meetings_jsons/meetings_2019.json"
]

In [17]:
#master list
extracted_old_meetings = []
#Opening each of the jsons
for file in json_files:
    if os.path.exists(file):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            #going through them
            for item in data.get("data", []):
                #getting the date
                date_dict = item.get("eli-dl:activity_date")
                #keeping only the date
                activity_date = date_dict.get("@value")[:10]

                old_meetings_dict = {
                    "id": item.get("id"),
                    "type": item.get("type"),
                    "activity_date": activity_date,
                    "activity_end_date": item.get("activity_end_date"),
                    "activity_id": item.get("activity_id"),
                    "activity_label": item.get("activity_label"),
                    "activity_start_date": item.get("activity_start_date"),
                    "consists_of": item.get("consists_of"),
                    "documented_by_a_realization_of": item.get("documented_by_a_realization_of"),
                    "had_activity_type": item.get("had_activity_type"),
                    "had_excused_person": item.get("had_excused_person"),
                    "had_participant_person": item.get("had_participant_person"),
                    "number_of_attendees": item.get("number_of_attendees"),
                    "parliamentary_term": item.get("parliamentary_term"),
                    "recorded_in_a_realization_of": item.get("recorded_in_a_realization_of"),
                    "hasLocality": item.get("hasLocality"),
                    "was_scheduled_in": item.get("was_scheduled_in")
                }

                #add to master list
                extracted_old_meetings.append(old_meetings_dict)

    else:
        print(f"file {file} not found.")

df_old_meetings_raw = pd.DataFrame(extracted_old_meetings)
print(f"Extracted {len(df_old_meetings_raw)} meetings from JSON files.")
df_old_meetings_raw.head(2)

Extracted 329 meetings from JSON files.


,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
0,eli/dl/event/MTG-PL-2014-01-13,Activity,2014-01-13,2014-01-13T23:00:00+01:00,MTG-PL-2014-01-13,"{'es': 'Lunes 13 de enero de 2014', 'el': 'Δευ...",2014-01-13T01:00:00+01:00,None,None,def/ep-activities/PLENARY_SITTING,None,None,NaN,org/ep-7,None,http://publications.europa.eu/resource/authori...,None
1,eli/dl/event/MTG-PL-2014-01-14,Activity,2014-01-14,2014-01-14T23:00:00+01:00,MTG-PL-2014-01-14,"{'el': 'Τρίτη 14 Ιανουαρίου 2014', 'sl': 'Tore...",2014-01-14T01:00:00+01:00,[eli/dl/event/MTG-PL-2014-01-14-VOT-ITM-344824...,None,def/ep-activities/PLENARY_SITTING,None,None,NaN,org/ep-7,None,http://publications.europa.eu/resource/authori...,None


In [21]:
df_meetings_full_decade = pd.concat([df, df_old_meetings_raw], ignore_index=True).drop(columns="Unnamed: 0")

In [23]:
print(len(df_meetings_full_decade))
df_meetings_full_decade.head(2)

544


,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
0,eli/dl/event/MTG-PL-2024-01-15,Activity,2024-01-15,2024-01-15T23:00:00+01:00,MTG-PL-2024-01-15,"{'hu': '2024. január 15., hétfő', 'sv': 'Månda...",2024-01-15T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-15-PVCRE-ITM-14'...,['eli/dl/doc/OJQ-9-2024-01-15'],def/ep-activities/PLENARY_SITTING,"['person/197683', 'person/197579', 'person/125...","['person/99283', 'person/88882', 'person/12480...",581.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-15-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN
1,eli/dl/event/MTG-PL-2024-01-16,Activity,2024-01-16,2024-01-16T23:00:00+01:00,MTG-PL-2024-01-16,"{'da': 'Tirsdag den 16. januar 2024', 'mt': ""I...",2024-01-16T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-16-PVCRE-ITM-11'...,['eli/dl/doc/OJQ-9-2024-01-16'],def/ep-activities/PLENARY_SITTING,"['person/36392', 'person/125030', 'person/1974...","['person/125214', 'person/197754', 'person/967...",636.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-16-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN


In [24]:
if "activity_id" in df_meetings_full_decade.columns:
    df_meetings_full_decade = df_meetings_full_decade.drop_duplicates(subset=["activity_id"], keep="last")

print(len(df_meetings_full_decade))

544


In [25]:
df_meetings_full_decade.to_csv("full_decade_raw_meetings.csv")

In [27]:
#Gonna clean the data frame from thursdays and mondays, both days when many are absent
#making sure the dates are datetime objects
df_meetings_full_decade["activity_date"] = pd.to_datetime(df_meetings_full_decade["activity_date"])

df_meetings_full_decade["weekday"] = df_meetings_full_decade["activity_date"].dt.day_name()
valid_days = ~df_meetings_full_decade["weekday"].isin(["Monday", "Thursday"])

df_midweek_meetings = df_meetings_full_decade[valid_days].copy()

print(f"Original meetings: {len(df_meetings_full_decade)}")
print(f"Midweek meetings: {len(df_midweek_meetings)}\n")
print(df_midweek_meetings.head(5))

Original meetings: 544
Midweek meetings: 280

                                id      type activity_date  \
1   eli/dl/event/MTG-PL-2024-01-16  Activity    2024-01-16   
2   eli/dl/event/MTG-PL-2024-01-17  Activity    2024-01-17   
6   eli/dl/event/MTG-PL-2024-02-06  Activity    2024-02-06   
7   eli/dl/event/MTG-PL-2024-02-07  Activity    2024-02-07   
10  eli/dl/event/MTG-PL-2024-02-27  Activity    2024-02-27   

            activity_end_date        activity_id  \
1   2024-01-16T23:00:00+01:00  MTG-PL-2024-01-16   
2   2024-01-17T23:00:00+01:00  MTG-PL-2024-01-17   
6   2024-02-06T23:00:00+01:00  MTG-PL-2024-02-06   
7   2024-02-07T23:00:00+01:00  MTG-PL-2024-02-07   
10  2024-02-27T23:00:00+01:00  MTG-PL-2024-02-27   

                                       activity_label  \
1   {'da': 'Tirsdag den 16. januar 2024', 'mt': "I...   
2   {'it': 'Mercoledì 17 gennaio 2024', 'en': 'Wed...   
6   {'hr': 'utorak, 6. veljače 2024.', 'cs': 'Úter...   
7   {'ro': 'Miercuri, 7 februarie 2024',

In [28]:
#SAVE DATA
df_midweek_meetings.to_csv("all_raw_midweek_meetings.csv")

In [5]:
df_midweek_meetings = pd.read_csv("all_raw_midweek_meetings.csv")

In [7]:
df_midweek_meetings.head(2)

,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in,weekday
0,1,eli/dl/event/MTG-PL-2024-01-16,Activity,2024-01-16,2024-01-16T23:00:00+01:00,MTG-PL-2024-01-16,"{'da': 'Tirsdag den 16. januar 2024', 'mt': ""I...",2024-01-16T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-16-PVCRE-ITM-11'...,['eli/dl/doc/OJQ-9-2024-01-16'],def/ep-activities/PLENARY_SITTING,"['person/36392', 'person/125030', 'person/1974...","['person/125214', 'person/197754', 'person/967...",636.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-16-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN,Tuesday
1,2,eli/dl/event/MTG-PL-2024-01-17,Activity,2024-01-17,2024-01-17T23:00:00+01:00,MTG-PL-2024-01-17,"{'it': 'Mercoledì 17 gennaio 2024', 'en': 'Wed...",2024-01-17T01:00:00+01:00,['eli/dl/event/MTG-PL-2024-01-17-VOT-ITM-93994...,['eli/dl/doc/OJQ-9-2024-01-17'],def/ep-activities/PLENARY_SITTING,"['person/198183', 'person/36392', 'person/9677...","['person/197607', 'person/197770', 'person/125...",637.0,org/ep-9,"['eli/dl/doc/PV-9-2024-01-17-ATT', 'eli/dl/doc...",http://publications.europa.eu/resource/authori...,NaN,Wednesday


In [9]:
#Now we get all the second readings of these meetings using the same pipeline as before
second_reading_weekdays = []
second_readings_json_folder = "second_reading_weekday_jsons"
os.makedirs(second_readings_json_folder, exist_ok=True)

# 1. Explicitly define your required headers, plus the language preference
headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0", 
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

# Define our magic search words (covering both new formatting and older French/English text)
#we make all search terms lowercase to make sure we don't miss anything cus of capitalization issues
second_reading_keywords = ["***ii", "*** ii", "second reading", "deuxième lecture", "deuxieme lecture", "deuxi\u00e8me lecture"]

for meeting_id in df_midweek_meetings["id"]:
    meeting_id = meeting_id.replace("eli/dl/event/", "")
    print(f"Scanning meeting: {meeting_id}...")
    url = f"https://data.europarl.europa.eu/api/v2/meetings/{meeting_id}/vote-results"

    
    try:
        # 2. Pass your master headers directly into the request
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()

            # Save the json dump
            try:
                # Sanitize the ID because it contains slashes (eli/dl/event/...)
                safe_filename = str(meeting_id).replace("/", "_") + ".json"
                file_path = os.path.join(second_readings_json_folder, safe_filename)
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, indent=4)
            except Exception as save_err:
                print(f"  -> Warning: Could not save file for {meeting_id}: {save_err}")
            
            if "data" in data and isinstance(data["data"], list):
                
                for vote_item in data["data"]:
                    
                    # Grab everything and convert to a lowercase string for easy searching
                    labels_str = str(vote_item.get("structuredLabel", "")).lower() + str(vote_item.get("activity_label", "")).lower()
                    
                    # Check if ANY of our magic keywords exist in the text
                    if any(keyword in labels_str for keyword in second_reading_keywords):
                        
                        item_id = vote_item.get("id", "Unknown ID")
                        
                        # Smart Title Grabber: Try English first, then fallback to French, then just grab whatever is there
                        final_title = "Title not found"
                        structured_label = vote_item.get("structuredLabel", {})
                        
                        if isinstance(structured_label, dict):
                            if "en" in structured_label:
                                final_title = structured_label["en"]
                            elif "fr" in structured_label:
                                final_title = structured_label["fr"]
                            elif len(structured_label) > 0:
                                # Just grab the first available translation
                                first_key = list(structured_label.keys())[0]
                                final_title = structured_label[first_key]
                        
                        print(f"\n🚨 FOUND ONE! Meeting: {meeting_id}")
                        print(f"ID: {item_id}")
                        print(f"Title: {final_title}\n")
                        
                        second_reading_weekdays.append({
                            "meeting_id": meeting_id,
                            "vot_itm_id": item_id,
                            "title": final_title
                        })
                        
        elif response.status_code == 204:
            pass # Silently skip empty days to keep the terminal clean
        else:
            print(f"Failed to fetch {meeting_id}: Status {response.status_code}")
            
    except Exception as e:
        print(f"Error on {meeting_id}: {e}")
        
    time.sleep(0.5)

df_weekday_second_readings = pd.DataFrame(second_reading_weekdays)

print(f"\n--- SCAN COMPLETE ---")
print(f"Total Second Readings found: {len(df_weekday_second_readings)}")
if not df_weekday_second_readings.empty:
    print(df_weekday_second_readings.head())

Scanning meeting: MTG-PL-2024-01-16...
Scanning meeting: MTG-PL-2024-01-17...
Scanning meeting: MTG-PL-2024-02-06...
Scanning meeting: MTG-PL-2024-02-07...
Scanning meeting: MTG-PL-2024-02-27...
Scanning meeting: MTG-PL-2024-02-28...
Scanning meeting: MTG-PL-2024-03-12...
Scanning meeting: MTG-PL-2024-03-13...
Scanning meeting: MTG-PL-2024-04-10...
Scanning meeting: MTG-PL-2024-04-23...
Scanning meeting: MTG-PL-2024-04-24...
Scanning meeting: MTG-PL-2024-07-16...
Scanning meeting: MTG-PL-2024-07-17...
Scanning meeting: MTG-PL-2024-07-19...
Scanning meeting: MTG-PL-2024-09-17...
Scanning meeting: MTG-PL-2024-09-18...
Scanning meeting: MTG-PL-2024-10-08...
Scanning meeting: MTG-PL-2024-10-09...
Scanning meeting: MTG-PL-2024-10-22...

🚨 FOUND ONE! Meeting: MTG-PL-2024-10-22
ID: eli/dl/event/MTG-PL-2024-10-22-VOT-ITM-962879
Title: <structuredLabel><title>Implementation of the Single European Sky (recast) ***II</title><label>Recommendation for second reading: Jens Gieseke, Johan Danielsson 

In [14]:
#save checkpoint csv
df_weekday_second_readings.to_csv("weekday_second_reading_meetings.csv")

In [ ]:
#Now we make new api requests using our json-dumps
#first we make it the right format
df_weekday_second_readings["vot_itm_id"] = df_weekday_second_readings["vot_itm_id"].str.replace("_", "/")

In [ ]:
votes_json_folder = "weekday_votes_json_dumps"
decisions_json_folder = "weekday_decisions_json_dumps"
docs_folder = "weekday_docs"

os.makedirs(votes_json_folder, exist_ok=True)
os.makedirs(decisions_json_folder, exist_ok=True)
os.makedirs(docs_folder, exist_ok=True)

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

def parse_older_vote_xml(xml_data, item_number):
    """Isolates the specific XML block for the item, extracts tags natively, and finds the outcome."""
    outcome_data = {
        "method": "Unknown",
        "outcome": "Unknown",
        "votes_favor": None,
        "votes_against": None,
        "abstentions": None
    }
    
    # 1. Grab everything inside the specific <Vote.Result Number="X."> tag
    pattern = rf'<Vote\.Result Number="{item_number}\.?">(.*?)</Vote\.Result>'
    match = re.search(pattern, xml_data, re.DOTALL | re.IGNORECASE)
    
    if not match:
        # If the item literally doesn't exist in the XML (like Item 64)
        return outcome_data
        
    raw_block = match.group(1)
    
    # 2. Extract Numbers BEFORE stripping tags (using the dedicated XML tags)
    # We use findall and grab the last one [-1] to ensure we get the final vote of the item
    for_matches = re.findall(r'<Vote\.Result\.Table\.TotalVote\.For>(\d+)<', raw_block)
    against_matches = re.findall(r'<Vote\.Result\.Table\.TotalVote\.Against>(\d+)<', raw_block)
    abs_matches = re.findall(r'<Vote\.Result\.Table\.TotalVote\.Abstention>(\d+)<', raw_block)
    
    if for_matches and against_matches and abs_matches:
        outcome_data["method"] = "Roll Call / Electronic"
        outcome_data["votes_favor"] = int(for_matches[-1])
        outcome_data["votes_against"] = int(against_matches[-1])
        outcome_data["abstentions"] = int(abs_matches[-1])
        
    # 3. Clean the XML tags out to evaluate the text outcome
    text_chunk = re.sub(r'<[^>]+>', ' ', raw_block)
    text_chunk = re.sub(r'\s+', ' ', text_chunk).strip()
    
    # 4. Determine Outcome
    if "Declared approved" in text_chunk or "Approval without vote" in text_chunk or "without vote" in text_chunk.lower():
        outcome_data["outcome"] = "Approved"
        # Only set to Auto-adopted if there was no roll-call vote on an amendment/rejection
        if outcome_data["method"] == "Unknown":
            outcome_data["method"] = "Auto-adopted"
    else:
        # Determine outcome based on isolated "+" or "-" symbols
        pluses = [m.start() for m in re.finditer(r'(?<=\s)\+(?=\s)', text_chunk)]
        minuses = [m.start() for m in re.finditer(r'(?<=\s)-(?=\s)', text_chunk)]
        
        last_plus = pluses[-1] if pluses else -1
        last_minus = minuses[-1] if minuses else -1
        
        if last_plus > last_minus:
            outcome_data["outcome"] = "Adopted"
        elif last_minus > last_plus:
            outcome_data["outcome"] = "Rejected"
        
    return outcome_data

all_extracted_data = []

for vote_id in df_weekday_second_readings["vot_itm_id"]:
    print(f"\nFetching: {vote_id}...")
    url_1 = f"https://data.europarl.europa.eu/{vote_id}"

    try:
        response_1 = requests.get(url_1, headers=headers, timeout=20)
        
        if response_1.status_code == 200:
            vote_data = response_1.json()

            safe_vote_filename = vote_id.replace("/", "_") + ".json"
            vote_file_path = os.path.join(votes_json_folder, safe_vote_filename)
            try:
                with open(vote_file_path, "w", encoding="utf-8") as f:
                    json.dump(vote_data, f, indent=4)
            except Exception:
                pass

            if "data" in vote_data and len(vote_data["data"]) > 0:
                vote_item = vote_data["data"][0]

                #set up the dataframe
                record = {
                    "vote_id": vote_id,
                    "decision_id": None,
                    "master_doc": None,
                    "item_number": None,
                    "start_date": vote_item.get("activity_start_date"),
                    "method": "Unknown",
                    "outcome": "Unknown",
                    "attendees": None,
                    "votes_favor": None,
                    "votes_against": None,
                    "abstentions": None,
                    "heading": None,
                    "absolute_majority": False
                }
                
                dec_found = False

                #If there is a DEC-code
                if "consists_of" in vote_item and isinstance(vote_item["consists_of"], list):
                    for decision_id in vote_item["consists_of"]:
                        if "-DEC-" in decision_id:
                            dec_found = True
                            print(f"  -> Routed to Modern API (DEC found)")
                            record["decision_id"] = decision_id
                            
                            url_2 = f"https://data.europarl.europa.eu/{decision_id}"
                            response_2 = requests.get(url_2, headers=headers, timeout=15)
                            
                            if response_2.status_code == 200:
                                decision_data = response_2.json()
                                
                                # Save the DEC file
                                safe_dec_filename = decision_id.split("/")[-1] + ".json"
                                with open(os.path.join(decisions_json_folder, safe_dec_filename), "w", encoding="utf-8") as f:
                                    json.dump(decision_data, f, indent=4)
                                
                                if "data" in decision_data and len(decision_data["data"]) > 0:
                                    target_dict = decision_data["data"][0]
                                    heading_en = target_dict.get("headingLabel", {}).get("en", "")
                                    
                                    # Populate the modern data
                                    record["method"] = target_dict.get("decision_method", "Unknown")
                                    record["outcome"] = target_dict.get("decision_outcome", "Unknown")
                                    record["attendees"] = target_dict.get("number_of_attendees")
                                    record["votes_favor"] = target_dict.get("number_of_votes_favor")
                                    record["votes_against"] = target_dict.get("number_of_votes_against")
                                    record["heading"] = heading_en
                            break

                #If no DEC-code
                if not dec_found:
                    pv_links = vote_item.get("recorded_in_a_realization_of", [])
                    if pv_links:
                        pv_item_link = pv_links[0]
                        print(f"  -> Routed to Historical XML Scraper (PV link found)")
                        
                        try:
                            # Split the link safely
                            if "-ITM-" in pv_item_link:
                                master_doc_id = pv_item_link.split("-ITM")[0]
                                raw_item_num = pv_item_link.split("-ITM-")[1]
                                item_number = str(int(raw_item_num))
                                
                                record["master_doc"] = master_doc_id
                                record["item_number"] = item_number
                                
                                # Fetch the Master Document JSON to find the XML URL
                                doc_url = f"https://data.europarl.europa.eu/{master_doc_id}"
                                doc_resp = requests.get(doc_url, headers=headers, timeout=15)
                                
                                if doc_resp.status_code == 200:
                                    master_doc_data = doc_resp.json()
                                    xml_download_path = None
                                    
                                    # Dig for the English XML link
                                    for item in master_doc_data.get("data", []):
                                        for realization in item.get("is_realized_by", []):
                                            if "language/ENG" in realization.get("language", ""):
                                                for embodiment in realization.get("is_embodied_by", []):
                                                    if "file-type/XML" in embodiment.get("format", ""):
                                                        xml_download_path = embodiment.get("is_exemplified_by")
                                                        break
                                                        
                                    if xml_download_path:
                                        # Download XML
                                        xml_url = f"https://data.europarl.europa.eu/{xml_download_path}"
                                        safe_doc_name = xml_download_path.split("/")[-1]
                                        local_doc_path = os.path.join(docs_folder, safe_doc_name)
                                        
                                        if not os.path.exists(local_doc_path):
                                            file_resp = requests.get(xml_url, timeout=30)
                                            with open(local_doc_path, "wb") as doc_file:
                                                doc_file.write(file_resp.content)
                                                
                                        # Parse XML
                                        with open(local_doc_path, "r", encoding="utf-8", errors="ignore") as xml_file:
                                            raw_xml = xml_file.read()
                                            
                                        result = parse_older_vote_xml(raw_xml, item_number)
                                        
                                        # Inject XML data into our unified record
                                        record["method"] = result["method"]
                                        record["outcome"] = result["outcome"]
                                        record["votes_favor"] = result["votes_favor"]
                                        record["votes_against"] = result["votes_against"]
                                        record["abstentions"] = result["abstentions"]
                        except Exception as pv_err:
                            print(f"  -> Error processing XML route: {pv_err}")
                    else:
                        print("  -> Dead End: No DEC and no PV linked.")
                        
                # 2. Append the finished record to our master list
                all_extracted_data.append(record)
                
        else:
            print(f"Failed to fetch {vote_id}: Status {response_1.status_code}")
            
    except requests.exceptions.Timeout:
        print(f"Warning: Request timed out for {vote_id}. Skipping...")
    except Exception as e:
        print(f"Error on {vote_id}: {e}")
        
    time.sleep(0.5)

df_weekday_vote_results = pd.DataFrame(all_extracted_data)
print("\n--- EXTRACTION COMPLETE ---")
print(f"Total processed: {len(df_weekday_vote_results)}")

if not df_weekday_vote_results.empty:
    print(df_weekday_vote_results[['vote_id', 'method', 'outcome', 'votes_favor', 'votes_against']].head())


Fetching: eli/dl/event/MTG-PL-2024-10-22-VOT-ITM-962879...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2024-12-17-VOT-ITM-965069...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965509...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965500...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965505...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-09-09-VOT-ITM-970636...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-10-07-VOT-ITM-973321...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974699...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974713...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/event/MTG-PL-2025-11-25-VOT-ITM-976302...
  -> Routed to Modern API (DEC found)

Fetching: eli/dl/ev

In [21]:
df_weekday_vote_results.sort_values("start_date", inplace=True)

In [22]:
df_weekday_vote_results.drop(columns="absolute_majority", inplace=True)

In [24]:
df_weekday_vote_results.isna().sum()

vote_id           0
decision_id      36
master_doc       18
item_number      18
start_date       54
method            0
outcome           0
attendees        48
votes_favor      31
votes_against    31
abstentions      37
heading          36
dtype: int64

In [27]:
def determine_winner(row):
    # Safely grab the outcome as an uppercase string
    outcome = str(row['outcome']).strip().upper()
    
    # 1. Explicit Approvals
    # This catches "Approval without vote" AND the "Approved" instances 
    # where a roll-call vote to reject the Council failed.
    if outcome in ["APPROVED", "APPROVAL WITHOUT VOTE"]:
        return "Council win"
        
    # 2. Modern API Labels & Historical +/- Parsing
    if "REJECTED" in outcome:
        # Parliament's motion to reject/amend failed
        return "Council win"
        
    if "ADOPTED" in outcome:
        # Parliament's motion to reject/amend succeeded
        return "Council loss"
        
    # 3. Mathematical Fail-Safe (based on your exact observation)
    # If the outcome text is ever missing or ambiguous, we check the raw numbers!
    v_fav = pd.to_numeric(row['votes_favor'], errors='coerce')
    v_agn = pd.to_numeric(row['votes_against'], errors='coerce')
    
    if pd.notna(v_fav) and pd.notna(v_agn):
        if v_agn > v_fav:
            return "Council win"  # The motion against the Council failed
        elif v_fav > v_agn:
            return "Council loss" # The motion against the Council succeeded
            
    return "Unknown"

# Apply the function across all rows (axis=1) in your merged dataframe
df_weekday_vote_results['Council win or loss'] = df_weekday_vote_results.apply(determine_winner, axis=1)

# Let's verify the results, specifically looking at the scenarios you highlighted!
print("=== OUTCOME DISTRIBUTION ===")
print(df_weekday_vote_results['Council win or loss'].value_counts(dropna=False))

print("\n=== VERIFYING YOUR SPECIFIC EDGE CASES ===")
# Show a sample of rows where there WAS a vote, but the Council still won
edge_cases = df_weekday_vote_results[(df_weekday_vote_results['method'].str.contains('Roll Call|ELECTRONIC', na=False, case=False)) & 
                      (df_weekday_vote_results['Council win or loss'] == 'Council win')]

print(edge_cases[['decision_id', 'heading', 'method', 'outcome', 'votes_favor', 'votes_against', 'Council win or loss']].head(10))

=== OUTCOME DISTRIBUTION ===
Council win or loss
Council win     38
Unknown         12
Council loss     4
Name: count, dtype: int64

=== VERIFYING YOUR SPECIFIC EDGE CASES ===
                                  decision_id heading  \
10  eli/dl/event/MTG-PL-2025-12-16-DEC-182608           
11  eli/dl/event/MTG-PL-2025-12-16-DEC-182611           
15  eli/dl/event/MTG-PL-2026-06-17-DEC-194440           
20                                        NaN     NaN   
21                                        NaN     NaN   
25                                        NaN     NaN   
26                                        NaN     NaN   
31                                        NaN     NaN   
32                                        NaN     NaN   
34                                        NaN     NaN   

                                              method  \
10  def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL   
11  def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL   
15  def/ep-decision-methods/

In [40]:
df_weekday_vote_results.head(60)

,vote_id,decision_id,master_doc,item_number,start_date,method,outcome,attendees,votes_favor,votes_against,abstentions,heading,Council win or loss
0,eli/dl/event/MTG-PL-2024-10-22-VOT-ITM-962879,eli/dl/event/MTG-PL-2024-10-22-DEC-170219,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
1,eli/dl/event/MTG-PL-2024-12-17-VOT-ITM-965069,eli/dl/event/MTG-PL-2024-12-17-DEC-171264,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
2,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965509,eli/dl/event/MTG-PL-2025-05-06-DEC-176083,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
3,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965500,eli/dl/event/MTG-PL-2025-05-06-DEC-176082,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
4,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965505,eli/dl/event/MTG-PL-2025-05-06-DEC-176084,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
5,eli/dl/event/MTG-PL-2025-09-09-VOT-ITM-970636,eli/dl/event/MTG-PL-2025-09-09-DEC-178814,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
6,eli/dl/event/MTG-PL-2025-10-07-VOT-ITM-973321,eli/dl/event/MTG-PL-2025-10-07-DEC-179299,NaN,NaN,None,def/ep-decision-methods/VOTE_HAND,def/ep-statuses/REJECTED,NaN,NaN,NaN,NaN,,Council win
7,eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974699,eli/dl/event/MTG-PL-2025-10-21-DEC-180204,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
8,eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974713,eli/dl/event/MTG-PL-2025-10-21-DEC-180203,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown
9,eli/dl/event/MTG-PL-2025-11-25-VOT-ITM-976302,eli/dl/event/MTG-PL-2025-11-25-DEC-182005,NaN,NaN,None,Unknown,Unknown,NaN,NaN,NaN,NaN,,Unknown


In [36]:
#Checking for edge cases where there is an absolute majority but still a council win
#Found one, but looking at the minutes from this meeting the roll call vote was not to reject
#so it is correct
df_weekday_vote_results[df_weekday_vote_results["votes_favor"] > 361]

,vote_id,decision_id,master_doc,item_number,start_date,method,outcome,attendees,votes_favor,votes_against,abstentions,heading,Council win or loss
12,eli/dl/event/MTG-PL-2026-01-21-VOT-ITM-981675,eli/dl/event/MTG-PL-2026-01-21-DEC-183557,NaN,NaN,None,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,656.0,632.0,15.0,NaN,,Council loss
17,eli/dl/event/MTG-PL-2026-07-07-VOT-ITM-992933,eli/dl/event/MTG-PL-2026-07-07-DEC-195007,NaN,NaN,None,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,661.0,646.0,12.0,NaN,,Council loss
25,eli/dl/event/MTG-PL-2014-04-16-VOT-ITM-352901-16,NaN,eli/dl/doc/PV-7-2014-04-16-VOT,16,None,Roll Call / Electronic,Approved,NaN,550.0,111.0,11.0,NaN,Council win
28,eli/dl/event/MTG-PL-2015-01-13-VOT-ITM-398848-5,NaN,eli/dl/doc/PV-8-2015-01-13-VOT,5,None,Roll Call / Electronic,Adopted,NaN,480.0,159.0,58.0,NaN,Council loss


In [42]:
#Let's check for other edge cases, where the parliament has had a single majority

mask = (361 > df_weekday_vote_results['votes_favor']) & (df_weekday_vote_results['votes_favor'] > df_weekday_vote_results['votes_against'])
df_filtered = df_weekday_vote_results[mask].copy()

df_filtered
#This is the vote to have an urgent vote on chat control. Simple majority. Should be removed

,vote_id,decision_id,master_doc,item_number,start_date,method,outcome,attendees,votes_favor,votes_against,abstentions,heading,Council win or loss
16,eli/dl/event/MTG-PL-2026-07-07-VOT-ITM-992951,eli/dl/event/MTG-PL-2026-07-07-DEC-195338,NaN,NaN,None,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,646.0,331.0,304.0,NaN,,Council loss


In [43]:
mask_unknown = (df_weekday_vote_results['method'] == 'Unknown') | (df_weekday_vote_results['outcome'] == 'Unknown')

print(f"Found {mask_unknown.sum()} unknown rows. Applying surgical fix...")

# 2. Loop through just those specific rows using .iterrows()
for idx, row in df_weekday_vote_results[mask_unknown].iterrows():
    
    # --- PATCH 1: Modern API (JSON) ---
    # If this row has a decision_id, it's a modern API vote
    if pd.notna(row.get('decision_id')):
        dec_id = row['decision_id']
        safe_dec_filename = dec_id.split("/")[-1] + ".json"
        dec_path = os.path.join("weekday_decisions_json_dumps", safe_dec_filename)
        
        if os.path.exists(dec_path):
            with open(dec_path, "r", encoding="utf-8") as f:
                dec_data = json.load(f)
                
                if "data" in dec_data and len(dec_data["data"]) > 0:
                    # Safely grab the reference text and force it to lowercase
                    ref_text = dec_data["data"][0].get("referenceText", {}).get("en", "").lower()
                    
                    if "approval without vote" in ref_text or "declared approved" in ref_text:
                        # .at[] updates the exact cell in your existing DataFrame instantly
                        df_weekday_vote_results.at[idx, 'outcome'] = "Approved"
                        df_weekday_vote_results.at[idx, 'method'] = "Auto-adopted"

    # --- PATCH 2: Historical Scraper (XML) ---
    # If this row has a master_doc, it's an older XML vote
    elif pd.notna(row.get('master_doc')) and pd.notna(row.get('item_number')):
        master_doc = row['master_doc']
        item_num = row['item_number']
        base_doc_name = master_doc.split("/")[-1]
        
        # Scan your local folder for the matching XML file
        for filename in os.listdir("weekday_docs"):
            if base_doc_name in filename and filename.endswith(".xml"):
                with open(os.path.join("weekday_docs", filename), "r", encoding="utf-8", errors="ignore") as f:
                    raw_xml = f.read()
                
                # Extract the specific vote block
                pattern = rf'<Vote\.Result Number="{item_num}\.?">(.*?)</Vote\.Result>'
                match = re.search(pattern, raw_xml, re.DOTALL | re.IGNORECASE)
                
                if match:
                    # Strip tags and force lowercase
                    text_chunk = re.sub(r'<[^>]+>', ' ', match.group(1))
                    text_chunk = re.sub(r'\s+', ' ', text_chunk).strip().lower()
                    
                    # Check for our keywords
                    if "declared approved" in text_chunk or "approval without vote" in text_chunk or "without vote" in text_chunk:
                        df_weekday_vote_results.at[idx, 'outcome'] = "Approved"
                        df_weekday_vote_results.at[idx, 'method'] = "Auto-adopted"
                break # Stop searching the folder once we found the file

print("Fix complete! Your dataframe has been updated in place.")

Found 13 unknown rows. Applying surgical fix...
Fix complete! Your dataframe has been updated in place.


In [44]:
df_weekday_vote_results

,vote_id,decision_id,master_doc,item_number,start_date,method,outcome,attendees,votes_favor,votes_against,abstentions,heading,Council win or loss
0,eli/dl/event/MTG-PL-2024-10-22-VOT-ITM-962879,eli/dl/event/MTG-PL-2024-10-22-DEC-170219,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
1,eli/dl/event/MTG-PL-2024-12-17-VOT-ITM-965069,eli/dl/event/MTG-PL-2024-12-17-DEC-171264,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
2,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965509,eli/dl/event/MTG-PL-2025-05-06-DEC-176083,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
3,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965500,eli/dl/event/MTG-PL-2025-05-06-DEC-176082,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
4,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965505,eli/dl/event/MTG-PL-2025-05-06-DEC-176084,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
5,eli/dl/event/MTG-PL-2025-09-09-VOT-ITM-970636,eli/dl/event/MTG-PL-2025-09-09-DEC-178814,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
6,eli/dl/event/MTG-PL-2025-10-07-VOT-ITM-973321,eli/dl/event/MTG-PL-2025-10-07-DEC-179299,NaN,NaN,None,def/ep-decision-methods/VOTE_HAND,def/ep-statuses/REJECTED,NaN,NaN,NaN,NaN,,Council win
7,eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974699,eli/dl/event/MTG-PL-2025-10-21-DEC-180204,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
8,eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974713,eli/dl/event/MTG-PL-2025-10-21-DEC-180203,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown
9,eli/dl/event/MTG-PL-2025-11-25-VOT-ITM-976302,eli/dl/event/MTG-PL-2025-11-25-DEC-182005,NaN,NaN,None,Auto-adopted,Approved,NaN,NaN,NaN,NaN,,Unknown


In [45]:
def determine_winner(row):
    # Safely grab the outcome as an uppercase string
    outcome = str(row['outcome']).strip().upper()
    
    # 1. Explicit Approvals
    # This catches "Approval without vote" AND the "Approved" instances 
    # where a roll-call vote to reject the Council failed.
    if outcome in ["APPROVED", "APPROVAL WITHOUT VOTE"]:
        return "Council win"
        
    # 2. Modern API Labels & Historical +/- Parsing
    if "REJECTED" in outcome:
        # Parliament's motion to reject/amend failed
        return "Council win"
        
    if "ADOPTED" in outcome:
        # Parliament's motion to reject/amend succeeded
        return "Council loss"
        
    # 3. Mathematical Fail-Safe (based on your exact observation)
    # If the outcome text is ever missing or ambiguous, we check the raw numbers!
    v_fav = pd.to_numeric(row['votes_favor'], errors='coerce')
    v_agn = pd.to_numeric(row['votes_against'], errors='coerce')
    
    if pd.notna(v_fav) and pd.notna(v_agn):
        if v_agn > v_fav:
            return "Council win"  # The motion against the Council failed
        elif v_fav > v_agn:
            return "Council loss" # The motion against the Council succeeded
            
    return "Unknown"

# Apply the function across all rows (axis=1) in your merged dataframe
df_weekday_vote_results['Council win or loss'] = df_weekday_vote_results.apply(determine_winner, axis=1)

# Let's verify the results, specifically looking at the scenarios you highlighted!
print("=== OUTCOME DISTRIBUTION ===")
print(df_weekday_vote_results['Council win or loss'].value_counts(dropna=False))

print("\n=== VERIFYING YOUR SPECIFIC EDGE CASES ===")
# Show a sample of rows where there WAS a vote, but the Council still won
edge_cases = df_weekday_vote_results[(df_weekday_vote_results['method'].str.contains('Roll Call|ELECTRONIC', na=False, case=False)) & 
                      (df_weekday_vote_results['Council win or loss'] == 'Council win')]

print(edge_cases[['decision_id', 'heading', 'method', 'outcome', 'votes_favor', 'votes_against', 'Council win or loss']].head(10))

=== OUTCOME DISTRIBUTION ===
Council win or loss
Council win     49
Council loss     4
Unknown          1
Name: count, dtype: int64

=== VERIFYING YOUR SPECIFIC EDGE CASES ===
                                  decision_id heading  \
10  eli/dl/event/MTG-PL-2025-12-16-DEC-182608           
11  eli/dl/event/MTG-PL-2025-12-16-DEC-182611           
15  eli/dl/event/MTG-PL-2026-06-17-DEC-194440           
20                                        NaN     NaN   
21                                        NaN     NaN   
25                                        NaN     NaN   
26                                        NaN     NaN   
31                                        NaN     NaN   
32                                        NaN     NaN   
34                                        NaN     NaN   

                                              method  \
10  def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL   
11  def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL   
15  def/ep-decision-methods/

In [ ]:
#remove the vote we found that was not an absolute majority vote
df_weekday_vote_results = df_weekday_vote_results[df_weekday_vote_results["decision_id"] != "eli/dl/event/MTG-PL-2026-07-07-DEC-195338"]

In [48]:
df_weekday_vote_results["Council win or loss"].value_counts(normalize=True, dropna=False) * 100

Council win or loss
Council win     92.452830
Council loss     5.660377
Unknown          1.886792
Name: proportion, dtype: float64

In [49]:
#Checkpoint csv
df_weekday_vote_results.to_csv("weekday_vote_results_council_wins_losses.csv")